# 01 LangChain LCEL 長文寫作基礎

這一章把「手動版 STORM」改成 LangChain v1 的標準寫法。重點不是多裝一個框架，而是把 prompt、model、parser 串成可讀、可測、可替換的 Runnable 管線。

## 學習目標

- 使用 `init_chat_model` 建立供應商無關模型。
- 用 `ChatPromptTemplate` 表達研究、大綱、寫作三個步驟。
- 用 LCEL 的 `prompt | model | parser` 建立長文寫作管線。
- 理解這一章如何銜接下一章 LangGraph 狀態圖。

## 1. 安裝與環境

課堂建議先在終端機安裝：

```bash
pip install langchain langchain-core langchain-openai python-dotenv pydantic
```

In [ ]:
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()
MODEL_NAME = os.getenv("COURSE_MODEL", "openai:gpt-4o-mini")
model = init_chat_model(MODEL_NAME)
parser = StrOutputParser()
print(f"使用模型: {MODEL_NAME}")

## 2. 建立研究摘要管線

STORM 的第一步是「先研究再寫」。這裡先用 prompt 模擬研究整理；之後會替換成 LangGraph deep research。

In [ ]:
research_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是研究編輯。請用繁體中文整理主題的關鍵背景、爭議、案例與可查證事實。"),
    ("human", "主題：{topic}\n目標讀者：{audience}\n請輸出條列研究摘要。"),
])

research_chain = research_prompt | model | parser

# 範例：取消註解即可呼叫模型
# research_notes = research_chain.invoke({"topic": "AI 對台灣中小企業的影響", "audience": "企業主管"})
# print(research_notes[:1000])

## 3. 建立大綱管線

大綱是長文品質的控制點。大綱先行，可以避免模型直接生成一大段失焦文章。

In [ ]:
outline_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是資深主編。根據研究摘要設計一份清楚、有層次的長文大綱。"),
    ("human", "主題：{topic}\n研究摘要：\n{research_notes}\n\n請輸出 Markdown 大綱，包含標題、前言、3-5 個主段落與結論。"),
])

outline_chain = outline_prompt | model | parser

## 4. 建立寫作管線

In [ ]:
writing_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是專業長文作者。請根據大綱與研究摘要撰寫清楚、有觀點、可讀性高的文章。"),
    ("human", "主題：{topic}\n目標讀者：{audience}\n研究摘要：\n{research_notes}\n\n大綱：\n{outline}\n\n請輸出完整 Markdown 文章。"),
])

writing_chain = writing_prompt | model | parser

## 5. 串成一個簡單流程

先用 Python function 包起來，下一章會把同樣流程改成 LangGraph 節點。

In [ ]:
def write_longform_article(topic: str, audience: str = "一般讀者") -> dict:
    research_notes = research_chain.invoke({"topic": topic, "audience": audience})
    outline = outline_chain.invoke({"topic": topic, "research_notes": research_notes})
    article = writing_chain.invoke({
        "topic": topic,
        "audience": audience,
        "research_notes": research_notes,
        "outline": outline,
    })
    return {
        "topic": topic,
        "audience": audience,
        "research_notes": research_notes,
        "outline": outline,
        "article": article,
    }

# result = write_longform_article("AI 對台灣中小企業的影響", "企業主管")
# print(result["article"][:1500])

## 小結

LCEL 適合直線流程：研究摘要 → 大綱 → 文章。當流程需要分支、狀態、多人協作、審核或重試，就進入下一章 LangGraph。